# Lab 2.4 · Interval coverage in the backtest

Day 2 · Session 4 (Intervals that hold) · 25 min.

An 80% interval is a promise: about 80 of 100 actual values fall inside. You measure whether three models
keep it, score their quantiles with pinball loss, and rebuild the intervals of the model that breaks its
promise from the errors it made in earlier folds.

**AI policy: AI-tutor.**

**Reference state.** Three supplied backtests of Al-Rimal over the 13 origins of 2025, each with quantile
columns: `data/reference/bt_stl_ets.csv`, `bt_arima_fourier.csv` and `bt_dynreg.csv`; and the regression's
backtest over the 13 origins of 2024, `bt_dynreg_2024.csv`. Nothing from your own earlier labs.

**Tasks**
- [ ] 1 · Compute 80% and 95% coverage for three models (5')
- [ ] 2 · Break coverage down by horizon week (4')
- [ ] 3 · Score each quantile with pinball loss (5')
- [ ] 4 · Rebuild the intervals from the errors of earlier folds (8')
- [ ] 5 · Report coverage before and after (3')
- [ ] ✱ Stretch · Check the single quantile the purchasing lead needs

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/forecasting.git"
    if not Path("/content/forecasting").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/forecasting"], check=True)
    os.chdir("/content/forecasting/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("shams.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import shams as U

plt = U.style()

BTS = {"decompose, then ETS": U.bt("stl_ets"), "ARIMA + annual Fourier": U.bt("arima_fourier"),
       "regression: weather + calendar": U.bt("dynreg")}
QS = list(U.QS)
print("columns:", list(BTS["regression: weather + calendar"].columns))
print({name: round(U.mase(b), 3) for name, b in BTS.items()})

## 1 · Coverage (5')

The 80% interval runs from the quantile 0.1 to the quantile 0.9; the 95% interval from 0.025 to 0.975.

✏️ **Exercise** · Return the share of rows whose actual value `y` lies inside `[lo, hi]`, where `lo` and
`hi` are column names.

In [ ]:
def coverage(bt: pd.DataFrame, lo: str, hi: str) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net: the cells below still run when the exercise is not done yet
def ref_coverage(bt, lo, hi):
    return float(((bt.y >= bt[lo]) & (bt.y <= bt[hi])).mean())


try:
    coverage(BTS["decompose, then ETS"], "q0.1", "q0.9")
except NotImplementedError:
    coverage = ref_coverage
cov = pd.DataFrame({name: {"80%": round(coverage(b, "q0.1", "q0.9"), 3), "95%": round(coverage(b, "q0.025", "q0.975"), 3)}
                    for name, b in BTS.items()}).T
print(cov.to_string())
U.check(abs(coverage(BTS["decompose, then ETS"], "q0.1", "q0.9") - U.coverage(BTS["decompose, then ETS"], 80)) < 1e-9, "coverage matches the helper function")
U.check(cov.loc["regression: weather + calendar", "80%"] < 0.70, "the most accurate model has the worst intervals: its 80% interval holds less than 70% of the time")

**Look at.** The regression has the best point score of the three and the narrowest promise kept. Its
intervals assume the future temperature is known. It is not.

## 2 · By horizon week (4')

✏️ **Exercise** · Return the 80% coverage for each week of the horizon (days 1-7, 8-14, 15-21, 22-28) as a
Series indexed 1 to 4.

In [ ]:
def coverage_by_week(bt: pd.DataFrame) -> pd.Series:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
def ref_coverage_by_week(bt):
    return ((bt.y >= bt["q0.1"]) & (bt.y <= bt["q0.9"])).groupby((bt.h - 1) // 7 + 1).mean()


try:
    coverage_by_week(BTS["decompose, then ETS"])
except NotImplementedError:
    coverage_by_week = ref_coverage_by_week
by_week = pd.DataFrame({name: coverage_by_week(b).round(3) for name, b in BTS.items()}).T
by_week.columns = [f"week {w}" for w in by_week.columns]
print(by_week.to_string())
U.check(by_week.shape == (3, 4), "one row per model, one column per horizon week")

## 3 · Pinball loss (5')

The pinball loss scores one quantile forecast `q` of level `tau`. When the actual value is above the
forecast it costs `tau` per unit; when it is below, `1 - tau` per unit. So a 0.9 quantile is punished nine
times as hard for being too low as for being too high. Lower is better. At `tau = 0.5` it is half the
absolute error.

✏️ **Exercise** · Return the mean pinball loss.

In [ ]:
def pinball(y, q, tau: float) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    pinball([10.0], [8.0], 0.9)
except NotImplementedError:
    pinball = U.pinball
U.check(abs(pinball([10.0], [8.0], 0.9) - 1.8) < 1e-9 and abs(pinball([10.0], [12.0], 0.9) - 0.2) < 1e-9,
        "a 0.9 quantile that is 2 too low costs 1.8; 2 too high costs 0.2")
pin = pd.DataFrame({name: {f"q{tau}": round(pinball(b.y / b["scale"], b[f"q{tau}"] / b["scale"], tau), 3) for tau in QS}
                    for name, b in BTS.items()}).T
pin["mean"] = pin.mean(axis=1).round(3)
print(pin.to_string())
U.check(pin["mean"].idxmin() == "ARIMA + annual Fourier", "by pinball loss the ranking changes: the best point forecast is not the best quantile forecast")

**Look at.** Compare the columns at the two ends. The regression pays most at `q0.025` and `q0.975`: its
extreme quantiles sit too close to the middle.

## 4 · Rebuild the intervals from earlier folds (8')

The backtest itself says how large the errors are, 1 to 28 days ahead, with everything that is not known at
the origin included. Use it:

1. For a fold, collect the scaled errors `(y - yhat) / scale` of all **earlier** folds, by horizon week.
2. Take their quantiles at 0.025, 0.1, 0.5, 0.9, 0.975.
3. The new quantile forecast is `yhat + error quantile * scale`.

Only earlier folds: the errors of a later fold are not known when this fold's forecast is made. And at
least a year of them: errors are larger in summer, so a calibration on winter folds alone would be too
narrow in July. `history` holds the same model's backtest over 2024 and 2025: 26 folds.

✏️ **Exercise** · Fill the new quantile columns for every fold whose origin is `test_from` or later, and
return those folds only.

In [ ]:
def rebuild(history: pd.DataFrame, test_from: pd.Timestamp, qs=QS) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
target = "regression: weather + calendar"
old = BTS[target]
history = pd.concat([U.bt("dynreg_2024"), old], ignore_index=True)
print(f"history: {history.origin.nunique()} folds, {history.date.min().date()} to {history.date.max().date()}")
try:
    new = rebuild(history, old.origin.min())
except NotImplementedError:
    new = U.bt("dynreg_rebuilt")
U.keep_bt("dynreg_rebuilt", new)
U.check(len(new) == 13 * 28, "the 13 folds of 2025, each calibrated on at least 13 earlier folds")
U.check(np.allclose(new.yhat, old.yhat.to_numpy()), "the point forecast is unchanged: only the quantiles moved")

## 5 · Before and after (3')

In [ ]:
report = pd.DataFrame({"model's own intervals": {"80% coverage": round(coverage(old, "q0.1", "q0.9"), 3),
                                                 "95% coverage": round(coverage(old, "q0.025", "q0.975"), 3),
                                                 "pinball (scaled)": round(U.scaled_pinball(old), 3),
                                                 "mean width of the 80% interval, MWh": round(float((old["q0.9"] - old["q0.1"]).mean()))},
                       "rebuilt from earlier folds": {"80% coverage": round(coverage(new, "q0.1", "q0.9"), 3),
                                                      "95% coverage": round(coverage(new, "q0.025", "q0.975"), 3),
                                                      "pinball (scaled)": round(U.scaled_pinball(new), 3),
                                                      "mean width of the 80% interval, MWh": round(float((new["q0.9"] - new["q0.1"]).mean()))}})
print(report.to_string())
U.check(abs(report.loc["80% coverage", "rebuilt from earlier folds"] - 0.8) < abs(report.loc["80% coverage", "model's own intervals"] - 0.8),
        "the rebuilt 80% interval is closer to its promise")
U.check(report.loc["pinball (scaled)", "rebuilt from earlier folds"] < report.loc["pinball (scaled)", "model's own intervals"],
        "and the pinball loss fell: wider, and better")

# The stretch, done: the purchasing lead's quantile, 0.8
try:
    new8 = rebuild(history, old.origin.min(), qs=[0.8])
    under = float((new8.y <= new8["q0.8"]).mean())
except NotImplementedError:
    under = U.reference("d2_4")["under_q80"]
print(f"demand stayed at or under the rebuilt 0.8 quantile on {under:.1%} of the test days")
new_by_week = ref_coverage_by_week(new).round(3)
print("rebuilt 80% coverage by horizon week:", new_by_week.tolist())
U.keep("d2_4", dict(coverage=cov.T.to_dict(), by_week=by_week.T.to_dict(), pinball=pin.T.to_dict(), report=report.to_dict(),
                    under_q80=under, new_by_week=new_by_week.tolist()))

**Look at.** The rebuilt interval is not exactly at 80% either, and by horizon week it still moves between
about 70% and 90%. Thirteen folds are 364 days: enough to see that an interval is broken and to repair
most of it, not enough to tune it to a decimal. Report the measured coverage as it is.

## Checkpoint

<details><summary>1 · Why were the regression's own intervals too narrow?</summary>
They are computed as if the future regressors were known. The calendar is; the temperature is not, and its
uncertainty is missing from the interval.</details>

<details><summary>2 · Why only the errors of earlier folds, and why a whole year of them?</summary>
An interval for a forecast made at an origin may only use what was known at that origin: errors of later
folds are in its future. A whole year, because the errors differ by season.</details>

<details><summary>3 · A model's 95% interval covers 99% of the days. Is that good?</summary>
No. It is too wide: the planner buys more reserve than the promise requires. Coverage should be close to
the stated level from both sides.</details>

## Carry forward

`bt_dynreg_rebuilt.csv` is the forecast with intervals that hold. Lab 4.4 draws its fan chart and quotes
its coverage in the planner's note.